# Discord notification test

This notebook checks that `notebook_notify.py` can format and send a Discord completion message.

Run the cells from top to bottom. The dry-run cell does not send anything; the final cell posts one test message to Discord.

In [1]:
from getpass import getpass
from pathlib import Path
from urllib import parse
import os
import sys

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "notebook_notify.py").exists()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebook_notify import notify_discord, save_discord_webhook_url
from notebook_notify import _resolve_webhook_url  # diagnostic-only helper

def mask_url(url):
    parsed = parse.urlparse(url)
    pieces = parsed.path.strip("/").split("/")
    if len(pieces) >= 3 and pieces[0] == "webhooks":
        return f"{parsed.scheme}://{parsed.netloc}/webhooks/{pieces[1][:4]}.../{pieces[2][:4]}..."
    return f"{parsed.scheme}://{parsed.netloc}/..."

saved_env_path = None

print(f"repo root: {root}")
print("env webhook configured:", bool(os.environ.get("DISCORD_WEBHOOK_URL")))
print(".env exists:", (root / ".env").exists())

try:
    resolved_webhook = _resolve_webhook_url(None)
except Exception as exc:
    print("webhook check failed:", exc)
    print("Paste the webhook URL below. It will be saved to .env for future runs.")
    pasted_webhook = getpass("Discord webhook URL: ").strip()
    if pasted_webhook:
        saved_env_path = save_discord_webhook_url(pasted_webhook, env_path=root / ".env")
        resolved_webhook = _resolve_webhook_url(None)
    else:
        resolved_webhook = None

if resolved_webhook:
    if not saved_env_path and not (root / ".env").exists():
        saved_env_path = save_discord_webhook_url(resolved_webhook, env_path=root / ".env")
    print("resolved webhook:", mask_url(resolved_webhook))
    if saved_env_path:
        print("saved to:", saved_env_path)
else:
    print("Fix: restart the notebook kernel after export, or create .env from .env.example.")

repo root: /app/Object_Detection
env webhook configured: False
.env exists: False
webhook check failed: Set DISCORD_WEBHOOK_URL or DISCORD_NOTEBOOK_WEBHOOK_URL, add it to .env, or pass webhook_url=...
Paste the webhook URL below. It will be saved to .env for future runs.


resolved webhook: https://discord.com/...
saved to: /app/Object_Detection/.env


In [2]:
result = notify_discord(
    "Dry-run check from discord_notify_test.ipynb. This does not send to Discord.",
    title="Discord notifier dry run",
    context={"notebook": "discord_notify_test.ipynb"},
    dry_run=True,
)
result

DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(), dry_run=True, error=None)

In [3]:
result = notify_discord(
    "Test message from Object_Detection notebook notification helper.",
    title="Discord notifier test",
    context={"notebook": "discord_notify_test.ipynb"},
    fail_silently=True,
)
if not result.ok:
    print("send failed:", result.error)
result

DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)